In [6]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


In [7]:
df = pd.read_csv("customer_support_tickets.csv")

In [8]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [9]:
df["full_text"] = df["Ticket Subject"] + " " + df["Ticket Description"]

In [10]:
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


#### Cleaning Function

In [11]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)


#### Apply Cleaning

In [12]:
df["clean_text"] = df["full_text"].apply(clean_text)

In [13]:
y_category = df["Ticket Type"]
y_priority = df["Ticket Priority"]

X = df["clean_text"]

In [14]:
vectorizer = TfidfVectorizer(max_features=5000)

X_vectorized = vectorizer.fit_transform(X)

In [15]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_vectorized, y_category, test_size=0.2, random_state=42
)

In [16]:
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_vectorized, y_priority, test_size=0.2, random_state=42
)

#### Category Model

In [17]:
category_model = LogisticRegression(max_iter=1000)

category_model.fit(X_train_c, y_train_c)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Priority Model

In [18]:
priority_model = LogisticRegression(max_iter=1000)

priority_model.fit(X_train_p, y_train_p)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Performance

In [19]:
y_pred_c = category_model.predict(X_test_c)

print("Category Classification Report:\n")
print(classification_report(y_test_c, y_pred_c))


Category Classification Report:

                      precision    recall  f1-score   support

     Billing inquiry       0.18      0.13      0.15       357
Cancellation request       0.17      0.18      0.17       327
     Product inquiry       0.19      0.19      0.19       316
      Refund request       0.19      0.21      0.20       345
     Technical issue       0.22      0.24      0.23       349

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.19      1694
        weighted avg       0.19      0.19      0.19      1694



In [20]:
y_pred_p = priority_model.predict(X_test_p)

print("Priority Classification Report:\n")
print(classification_report(y_test_p, y_pred_p))


Priority Classification Report:

              precision    recall  f1-score   support

    Critical       0.27      0.29      0.28       411
        High       0.26      0.27      0.27       409
         Low       0.25      0.24      0.24       415
      Medium       0.28      0.27      0.28       459

    accuracy                           0.27      1694
   macro avg       0.27      0.27      0.27      1694
weighted avg       0.27      0.27      0.27      1694



In [21]:
print(confusion_matrix(y_test_c, y_pred_c))


[[46 69 71 91 80]
 [42 58 73 71 83]
 [49 65 60 71 71]
 [63 72 64 74 72]
 [54 82 54 75 84]]


In [22]:
joblib.dump(category_model, "category_model.pkl")
joblib.dump(priority_model, "priority_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")


['vectorizer.pkl']

#### Prediction Function

In [23]:
def predict_ticket(text):

    cleaned = clean_text(text)
    vect = vectorizer.transform([cleaned])

    category = category_model.predict(vect)[0]
    priority = priority_model.predict(vect)[0]

    return category, priority


In [24]:
ticket = "Payment deducted but order not confirmed"

predict_ticket(ticket)

('Refund request', 'Medium')